# Baseline Model PII Detection

Notebook ini digunakan untuk melatih dan mengevaluasi model baseline **Logistic Regression** dan **Linear SVM** untuk deteksi Personally Identifiable Information (PII) pada level token.

Alur utama:
1. Load dataset hasil preprocessing.
2. Ekstraksi fitur token.
3. Ubah fitur menjadi representasi numerik.
4. Train Logistic Regression dan Linear SVM.
5. Simpan prediksi, metrik, dan model.
6. Visualisasi perbandingan performa.

## 1. Import Library

Bagian ini memanggil library yang dibutuhkan untuk membaca data, membuat fitur, melakukan vectorization, melatih model, menyimpan model, dan menampilkan grafik evaluasi.

In [ ]:
import os
import sys
import csv
import re
import time
import json
import joblib
import warnings

import matplotlib.pyplot as plt

from scipy.sparse import hstack
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. Path Project

Notebook ini diasumsikan dijalankan dari root project. Jika notebook diletakkan di folder lain, sesuaikan nilai `BASE_DIR`.

In [ ]:
BASE_DIR = os.getcwd()
sys.path.append(BASE_DIR)

TRAIN_PATH = os.path.join(BASE_DIR, "data", "processed", "train.json")
VAL_PATH = os.path.join(BASE_DIR, "data", "processed", "val.json")
TEST_PATH = os.path.join(BASE_DIR, "data", "processed", "test_internal.json")

MODEL_DIR = os.path.join(BASE_DIR, "models", "baseline")
PRED_DIR = os.path.join(BASE_DIR, "results", "predictions")
METRIC_DIR = os.path.join(BASE_DIR, "results", "metrics")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(METRIC_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)

## 3. Load Helper dari Project

`DataLoader`, `evaluate_from_csv`, dan `save_metrics` diambil dari folder `src`. Fungsi ini mengikuti struktur project yang sudah ada.

In [ ]:
from src.data_loader import DataLoader
from src.evaluate import evaluate_from_csv, save_metrics

## 4. Fungsi Utility

Fungsi ini dipakai untuk menampilkan progress training dan menghitung waktu eksekusi.

In [ ]:
def progress(message):
    print(message, flush=True)


def elapsed(start_time):
    return f"{time.time() - start_time:.2f}s"

## 5. Feature Engineering

Setiap token diubah menjadi fitur seperti panjang token, pola huruf kapital, angka, email, URL, prefix, suffix, serta token sebelum dan sesudahnya. Fitur konteks digunakan karena Logistic Regression dan Linear SVM tidak memodelkan urutan token secara langsung seperti CRF.

In [ ]:
def is_email(token):
    return bool(re.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", token))


def is_url(token):
    token = token.lower()
    return (
        "http" in token
        or "www" in token
        or ".com" in token
        or ".org" in token
        or ".net" in token
    )


def token_shape(token):
    result = ""

    for char in token:
        if char.isupper():
            result += "X"
        elif char.islower():
            result += "x"
        elif char.isdigit():
            result += "d"
        else:
            result += char

    return result


def extract_features(tokens, i):
    token = tokens[i]
    lower = token.lower()

    features = {
        "token_lower": lower,
        "token_shape": token_shape(token),
        "token_length": len(token),
        "is_digit": token.isdigit(),
        "is_upper": token.isupper(),
        "is_title": token.istitle(),
        "has_digit": any(char.isdigit() for char in token),
        "has_alpha": any(char.isalpha() for char in token),
        "has_at": "@" in token,
        "has_dot": "." in token,
        "has_dash": "-" in token,
        "is_email": is_email(token),
        "is_url": is_url(token),
        "prefix_1": lower[:1],
        "prefix_2": lower[:2],
        "prefix_3": lower[:3],
        "suffix_1": lower[-1:],
        "suffix_2": lower[-2:],
        "suffix_3": lower[-3:],
    }

    if i > 0:
        prev_token = tokens[i - 1]
        features["prev_token"] = prev_token.lower()
        features["prev_shape"] = token_shape(prev_token)
        features["prev_is_title"] = prev_token.istitle()
    else:
        features["BOS"] = True

    if i < len(tokens) - 1:
        next_token = tokens[i + 1]
        features["next_token"] = next_token.lower()
        features["next_shape"] = token_shape(next_token)
        features["next_is_title"] = next_token.istitle()
    else:
        features["EOS"] = True

    return features

## 6. Persiapan Dataset

Data dokumen diubah menjadi data token-level. Setiap token memiliki fitur, label asli, dan metadata untuk menyimpan hasil prediksi ke CSV.

In [ ]:
def prepare_data(data, dataset_name):
    X_features = []
    X_tokens = []
    y = []
    meta = []

    total_docs = len(data)
    start_time = time.time()

    progress(f"Preparing {dataset_name} data...")
    progress(f"{dataset_name}: total documents = {total_docs}")

    for doc_idx, doc in enumerate(data, start=1):
        doc_id = doc["document"]
        tokens = doc["tokens"]
        labels = doc["labels"]

        for i, token in enumerate(tokens):
            X_features.append(extract_features(tokens, i))
            X_tokens.append(token.lower())
            y.append(labels[i])
            meta.append({
                "document_id": doc_id,
                "token": token,
                "true_label": labels[i],
            })

        if doc_idx % 500 == 0 or doc_idx == total_docs:
            progress(
                f"{dataset_name}: processed {doc_idx}/{total_docs} documents "
                f"| tokens = {len(X_tokens)} "
                f"| elapsed = {elapsed(start_time)}"
            )

    progress(f"{dataset_name}: feature extraction done")
    progress(f"{dataset_name}: total tokens = {len(X_tokens)}")

    return X_features, X_tokens, y, meta

## 7. Load Data

Dataset yang digunakan adalah `train.json`, `val.json`, dan `test_internal.json`. Evaluasi akhir dilakukan pada `test_internal.json`.

In [ ]:
loader = DataLoader()

train_data = loader.load_raw_json(TRAIN_PATH)
val_data = loader.load_raw_json(VAL_PATH)
test_data = loader.load_raw_json(TEST_PATH)

print("Train documents:", len(train_data))
print("Validation documents:", len(val_data))
print("Test internal documents:", len(test_data))

## 8. Ekstraksi Fitur

Pada tahap ini, seluruh token dari dataset train, validation, dan test diubah menjadi fitur handcrafted dan daftar token lowercase.

In [ ]:
X_train_features, X_train_tokens, y_train, _ = prepare_data(train_data, "train")
X_val_features, X_val_tokens, _, _ = prepare_data(val_data, "val")
X_test_features, X_test_tokens, _, meta_test = prepare_data(test_data, "test_internal")

## 9. Vectorization

Fitur dictionary diubah menggunakan `DictVectorizer`, sedangkan teks token diubah menggunakan TF-IDF character n-gram. Character n-gram membantu model menangkap pola seperti email, URL, nama, dan nomor identitas.

In [ ]:
def build_feature_matrix(dict_vectorizer, tfidf_vectorizer, X_features, X_tokens, fit=False):
    if fit:
        dict_matrix = dict_vectorizer.fit_transform(X_features)
        tfidf_matrix = tfidf_vectorizer.fit_transform(X_tokens)
    else:
        dict_matrix = dict_vectorizer.transform(X_features)
        tfidf_matrix = tfidf_vectorizer.transform(X_tokens)

    return hstack([dict_matrix, tfidf_matrix])


dict_vectorizer = DictVectorizer(sparse=True)
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    min_df=2
)

vector_start = time.time()

X_train = build_feature_matrix(dict_vectorizer, tfidf_vectorizer, X_train_features, X_train_tokens, fit=True)
X_val = build_feature_matrix(dict_vectorizer, tfidf_vectorizer, X_val_features, X_val_tokens, fit=False)
X_test = build_feature_matrix(dict_vectorizer, tfidf_vectorizer, X_test_features, X_test_tokens, fit=False)

print("Train matrix shape:", X_train.shape)
print("Val matrix shape:", X_val.shape)
print("Test matrix shape:", X_test.shape)
print("Vectorizing finished in", elapsed(vector_start))

## 10. Fungsi Simpan Prediksi dan Training

Fungsi berikut digunakan untuk melatih model, melakukan prediksi, menyimpan hasil prediksi, menghitung metrik, dan menyimpan model dalam format `.joblib`.

In [ ]:
def save_predictions(meta, predictions, output_path):
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["document_id", "token", "true_label", "pred_label"]
        )

        writer.writeheader()

        for row, pred in zip(meta, predictions):
            writer.writerow({
                "document_id": row["document_id"],
                "token": row["token"],
                "true_label": row["true_label"],
                "pred_label": pred,
            })


def print_simple_metrics(metrics):
    token = metrics["token_level"]
    entity = metrics["entity_level"]

    print(f"{metrics['model']} result:")
    print(f"total tokens: {metrics['total_tokens']}")

    print(
        f"token-level: "
        f"precision = {token['precision']:.4f}, "
        f"recall = {token['recall']:.4f}, "
        f"f1 = {token['f1']:.4f}"
    )

    print(
        f"entity-level: "
        f"precision = {entity['precision']:.4f}, "
        f"recall = {entity['recall']:.4f}, "
        f"f1 = {entity['f1']:.4f}"
    )


def train_model(model_name, classifier, X_train, y_train, X_test, meta_test):
    start_time = time.time()

    print(f"\n{model_name}: training started")
    classifier.fit(X_train, y_train)
    print(f"{model_name}: training finished in {elapsed(start_time)}")

    predictions = classifier.predict(X_test)

    pred_path = os.path.join(PRED_DIR, f"{model_name}_predictions.csv")
    save_predictions(meta_test, predictions, pred_path)

    metrics = evaluate_from_csv(pred_path, model_name)

    metric_path = os.path.join(METRIC_DIR, f"{model_name}_metrics.json")
    save_metrics(metrics, metric_path)

    model_object = {
        "model_name": model_name,
        "classifier": classifier,
        "dict_vectorizer": dict_vectorizer,
        "tfidf_vectorizer": tfidf_vectorizer,
        "features": [
            "token length",
            "character features",
            "capitalization pattern",
            "digit pattern",
            "email pattern",
            "URL pattern",
            "prefix and suffix",
            "previous token",
            "next token",
            "TF-IDF token representation",
        ],
    }

    model_path = os.path.join(MODEL_DIR, f"{model_name}_model.joblib")
    joblib.dump(model_object, model_path)

    print(f"{model_name}: predictions saved to {pred_path}")
    print(f"{model_name}: metrics saved to {metric_path}")
    print(f"{model_name}: model saved to {model_path}")

    print_simple_metrics(metrics)
    return metrics

## 11. Training Logistic Regression

Logistic Regression menggunakan `class_weight="balanced"` untuk membantu menangani distribusi kelas yang tidak sepenuhnya merata. Model ini cenderung memiliki recall tinggi, tetapi perlu diperhatikan precision-nya.

In [ ]:
logistic_regression = LogisticRegression(
    class_weight="balanced",
    max_iter=150,
    solver="saga",
    tol=1e-3,
    verbose=0
)

logreg_metrics = train_model(
    "logistic_regression",
    logistic_regression,
    X_train,
    y_train,
    X_test,
    meta_test
)

## 12. Training Linear SVM

Linear SVM digunakan sebagai baseline pembanding karena cocok untuk data sparse berdimensi tinggi seperti gabungan fitur handcrafted dan TF-IDF.

In [ ]:
linear_svm = LinearSVC(
    class_weight="balanced",
    max_iter=1000,
    tol=1e-3
)

svm_metrics = train_model(
    "linear_svm",
    linear_svm,
    X_train,
    y_train,
    X_test,
    meta_test
)

## 13. Visualisasi Hasil

Grafik berikut membandingkan precision, recall, dan F1-score untuk Logistic Regression dan Linear SVM pada level token dan level entitas.

In [ ]:
def plot_comparison(logreg_metrics, svm_metrics, level, title):
    metrics_name = ["precision", "recall", "f1"]
    x = range(len(metrics_name))

    logreg_scores = [logreg_metrics[level][m] for m in metrics_name]
    svm_scores = [svm_metrics[level][m] for m in metrics_name]

    width = 0.35

    plt.figure(figsize=(10, 5))
    plt.bar([i - width/2 for i in x], logreg_scores, width, label="Logistic Regression")
    plt.bar([i + width/2 for i in x], svm_scores, width, label="Linear SVM")

    plt.xticks(list(x), ["Precision", "Recall", "F1-Score"])
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plt.title(title)
    plt.legend()

    for i, score in enumerate(logreg_scores):
        plt.text(i - width/2, score + 0.02, f"{score:.3f}", ha="center")

    for i, score in enumerate(svm_scores):
        plt.text(i + width/2, score + 0.02, f"{score:.3f}", ha="center")

    plt.show()


plot_comparison(logreg_metrics, svm_metrics, "token_level", "Token-Level Performance Comparison")
plot_comparison(logreg_metrics, svm_metrics, "entity_level", "Entity-Level Performance Comparison")

## 14. Analisis Singkat

Berdasarkan hasil evaluasi, Logistic Regression cenderung memiliki recall yang tinggi, sehingga model mampu mendeteksi banyak token PII. Namun, precision-nya lebih rendah dibandingkan Linear SVM, sehingga masih terdapat lebih banyak false positive.

Linear SVM menunjukkan performa yang lebih seimbang antara precision dan recall. Oleh karena itu, Linear SVM dapat dijadikan baseline utama untuk dibandingkan dengan model yang lebih kompleks seperti CRF, boosting, ensemble, atau deep learning.

## 15. Output yang Dihasilkan

Setelah notebook dijalankan, file yang dihasilkan adalah:

```text
results/predictions/logistic_regression_predictions.csv
results/predictions/linear_svm_predictions.csv
results/metrics/logistic_regression_metrics.json
results/metrics/linear_svm_metrics.json
models/baseline/logistic_regression_model.joblib
models/baseline/linear_svm_model.joblib
```